# Сервис статистики SQLite

In [1]:
from datetime import datetime
from pathlib import Path
from xmlrpc.server import SimpleXMLRPCServer, SimpleXMLRPCRequestHandler
import sqlite3

DB_PATH = Path('log.db')
STATS_HOST = '127.0.0.1'
STATS_PORT = 8019

class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)


def connect_db():
    connection = sqlite3.connect(DB_PATH, timeout=5.0)
    connection.row_factory = sqlite3.Row
    return connection


def init_db():
    with connect_db() as connection:
        connection.execute("""CREATE TABLE IF NOT EXISTS events (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            event_time TEXT NOT NULL,
            event_type TEXT NOT NULL,
            operation TEXT NOT NULL,
            wait_seconds REAL,
            execution_seconds REAL,
            duration_seconds REAL,
            status TEXT NOT NULL
        )""")
        fields = {row['name'] for row in connection.execute('PRAGMA table_info(events)')}
        if 'server_execution_seconds' in fields:
            try:
                connection.execute('ALTER TABLE events DROP COLUMN server_execution_seconds')
            except sqlite3.OperationalError:
                connection.execute("""CREATE TABLE events_migrated (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    event_time TEXT NOT NULL,
                    event_type TEXT NOT NULL,
                    operation TEXT NOT NULL,
                    wait_seconds REAL,
                    execution_seconds REAL,
                    duration_seconds REAL,
                    status TEXT NOT NULL
                )""")
                connection.execute("""INSERT INTO events_migrated
                    (id, event_time, event_type, operation, wait_seconds,
                     execution_seconds, duration_seconds, status)
                    SELECT id, event_time, event_type, operation, wait_seconds,
                           execution_seconds, duration_seconds, status FROM events""")
                connection.execute('DROP TABLE events')
                connection.execute('ALTER TABLE events_migrated RENAME TO events')
        connection.execute('CREATE INDEX IF NOT EXISTS idx_event_time ON events(event_time)')
        connection.execute('CREATE INDEX IF NOT EXISTS idx_operation ON events(operation)')
        connection.execute('CREATE INDEX IF NOT EXISTS idx_duration ON events(duration_seconds)')


init_db()

## Приём и фильтрация событий

In [2]:
def add_log(event_time, event_type, operation, wait_seconds=None,
            execution_seconds=None, duration_seconds=None, status='success'):
    datetime.fromisoformat(event_time)
    if not event_type or not operation:
        raise ValueError('Не указан тип события или название операции')

    with connect_db() as connection:
        connection.execute(
            """INSERT INTO events (
                event_time, event_type, operation, wait_seconds,
                execution_seconds, duration_seconds, status
            ) VALUES (?, ?, ?, ?, ?, ?, ?)""",
            (
                event_time, event_type, operation, wait_seconds,
                execution_seconds, duration_seconds, status
            )
        )
    return True


def get_log(operation='', time_from='', time_to='', duration_from=None,
            duration_to=None, event_type='', limit=500):
    if time_from:
        datetime.fromisoformat(time_from)
    if time_to:
        datetime.fromisoformat(time_to)
    if time_from and time_to and time_from > time_to:
        raise ValueError('Начало интервала позже конца')
    if duration_from is not None and duration_to is not None and duration_from > duration_to:
        raise ValueError('Нижняя граница длительности больше верхней')

    conditions = []
    params = []

    if operation:
        conditions.append('operation = ?')
        params.append(operation)
    if time_from:
        conditions.append('event_time >= ?')
        params.append(time_from)
    if time_to:
        conditions.append('event_time <= ?')
        params.append(time_to)
    if duration_from is not None:
        conditions.append('duration_seconds >= ?')
        params.append(float(duration_from))
    if duration_to is not None:
        conditions.append('duration_seconds <= ?')
        params.append(float(duration_to))
    if event_type:
        conditions.append('event_type = ?')
        params.append(event_type)

    query = 'SELECT * FROM events'
    if conditions:
        query += ' WHERE ' + ' AND '.join(conditions)
    query += ' ORDER BY id DESC LIMIT ?'
    params.append(min(max(1, int(limit)), 5000))

    with connect_db() as connection:
        rows = [dict(row) for row in connection.execute(query, params).fetchall()]

    return rows[::-1]


def get_status():
    with connect_db() as connection:
        count = connection.execute('SELECT COUNT(*) FROM events').fetchone()[0]
        failed = connection.execute(
            "SELECT COUNT(*) FROM events WHERE status = 'error'"
        ).fetchone()[0]
        limited = connection.execute(
            "SELECT COUNT(*) FROM events WHERE event_type = 'too_many_requests'"
        ).fetchone()[0]
    return {
        'database': str(DB_PATH),
        'total_events': count,
        'errors': failed,
        'rate_limit_events': limited,
    }

## Запуск сервера статистики

In [ ]:
server = SimpleXMLRPCServer(
    (STATS_HOST, STATS_PORT),
    requestHandler=RequestHandler,
    allow_none=True,
    logRequests=False
)
server.register_function(add_log, 'add_log')
server.register_function(get_log, 'get_log')
server.register_function(get_status, 'get_status')
server.register_introspection_functions()

print(f'Сервер статистики: http://{STATS_HOST}:{STATS_PORT}/RPC2', flush=True)
print(f'База данных: {DB_PATH.resolve()}', flush=True)
server.serve_forever()

Сервер статистики: http://127.0.0.1:8019/RPC2
База данных: C:\Users\manoh\Desktop\Учеба\Магистратура\1 семестр\РСОИ\xmlrpc-labs\log.db
